Using media stack api

In [2]:
import os
import http.client
import urllib.parse
import json
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

MEDIA_STACK_API_KEY = os.getenv("MEDIA_STACK_API_KEY")

conn = http.client.HTTPConnection('api.mediastack.com')

params = urllib.parse.urlencode({
    'access_key': MEDIA_STACK_API_KEY,
    'categories': '-general,-sports',
    'sort': 'published_desc',
    'limit': 10,
})

conn.request('GET', '/v1/news?{}'.format(params))

res = conn.getresponse()
data = res.read()

# Convert byte response to JSON
json_data = json.loads(data.decode('utf-8'))

# Now json_data is a Python dictionary
print(json.dumps(json_data, indent=2))  # Pretty-print the JSON

{
  "pagination": {
    "limit": 10,
    "offset": 0,
    "count": 10,
    "total": 10000
  },
  "data": [
    {
      "author": null,
      "title": "Huntington Ingalls reports mixed Q1 results; reaffirms FY25 outlook",
      "description": "Huntington Ingalls reports mixed Q1 results; reaffirms FY25 outlook",
      "url": "https://seekingalpha.com/news/4438855-huntington-ingalls-reports-mixed-q1-results-reaffirms-fy25-outlook?utm_source=feed_news_all&utm_medium=referral&feed_item_type=news",
      "source": "Seeking Alpha",
      "image": null,
      "category": "business",
      "language": "en",
      "country": "us",
      "published_at": "2025-05-01T11:20:23+00:00"
    },
    {
      "author": "zee business",
      "title": "Helicopter service for Kedarnath Yatra 2025 begins from Sonprayag; online tickets available on IRCTC website",
      "description": "Pawan Rana, a helicopter operator, said, \"Online tickets are available on IRCTC, while offline tickets can be obtained throug

Using a general html parser to get the content

In [5]:

import requests
from newspaper import Article

url = 'https://seekingalpha.com/news/4438841-alkermes-gaap-eps-of-0_13-beats-by-0_08-revenue-of-306_5m-beats-by-2_39m?utm_source=feed_news_all&utm_medium=referral&feed_item_type=news'
article = Article(url)

## Download HTML yourself and insert into Newspaper3k
response = requests.get(url)
article.download(input_html=response.text)


In [12]:
import json
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from newspaper import Article

def scrape_with_newspaper(url):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("user-agent=Mozilla/5.0")

    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(3)

    html = driver.page_source
    driver.quit()

    article = Article(url)
    article.set_html(html)
    article.download_state = 2  # Force state to SUCCESS
    article.parse()

    result = {
        "title": article.title,
        "url": url,
        "authors": article.authors,
        "published_date": article.publish_date.isoformat() if article.publish_date else None,
        "paragraphs": [p.strip() for p in article.text.split("\n") if p.strip()]
    }

    print(json.dumps(result, indent=2))

# Example usage
scrape_with_newspaper("https://seekingalpha.com/news/4438841-alkermes-gaap-eps-of-0_13-beats-by-0_08-revenue-of-306_5m-beats-by-2_39m")


{
  "title": "Alkermes GAAP EPS of $0.13 beats by $0.08, revenue of $306.5M beats by $2.39M (NASDAQ:ALKS)",
  "url": "https://seekingalpha.com/news/4438841-alkermes-gaap-eps-of-0_13-beats-by-0_08-revenue-of-306_5m-beats-by-2_39m",
  "authors": [],
  "published_date": null,
  "paragraphs": [
    "Search field Entering text into the input field will update the search result below",
    "Entering text into the input field will update the search result below"
  ]
}


Using news paper3k and selenium

In [10]:
import json
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from newspaper import Article

def scrape_with_newspaper3k(url):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("user-agent=Mozilla/5.0")

    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(3)

    html = driver.page_source
    driver.quit()

    article = Article(url)
    article.set_html(html)
    article.download_state = 2  # Force state to SUCCESS
    article.parse()

    result = {
        "title": article.title,
        "url": url,
        "authors": article.authors,
        "published_date": article.publish_date.isoformat() if article.publish_date else None,
        "paragraphs": [p.strip() for p in article.text.split("\n") if p.strip()]
    }

    print(json.dumps(result, indent=2))

# Example usage
scrape_with_newspaper3k("https://www.thehindubusinessline.com/companies/adani-enterprises-fourth-quarter-profit-drops-on-coal-trading-weakness/article69521415.ece")


{
  "title": "Access to this page has been denied",
  "url": "https://seekingalpha.com/news/4438841-alkermes-gaap-eps-of-0_13-beats-by-0_08-revenue-of-306_5m-beats-by-2_39m",
  "authors": [],
  "published_date": null,
  "paragraphs": [
    "Before we continue...",
    "Press & Hold to confirm you are",
    "a human (and not a bot).",
    "Reference ID d54b5074-2688-11f0-8dee-31d643caa8b9"
  ]
}


In [17]:
import json
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# Custom headers and cookies
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}

COOKIES = {
    'name_of_the_consent_cookie': 'value_indicating_consent',
}

def scrape_with_bs4(url):
    options = Options()
    options.add_argument("--headless")
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")

    driver = webdriver.Chrome(options=options)
    driver.get(url)

    # Inject cookies before page loads (optional depending on site)
    for name, value in COOKIES.items():
        driver.add_cookie({'name': name, 'value': value, 'domain': '.ndtvprofit.com'})

    driver.get(url)  # Reload page with cookies now set
    time.sleep(3)

    html = driver.page_source
    driver.quit()

    soup = BeautifulSoup(html, "html.parser")

    # Extract title
    title_tag = soup.find("h1") or soup.title
    title = title_tag.get_text(strip=True) if title_tag else "N/A"

    # Extract paragraphs
    paragraphs = [p.get_text(strip=True) for p in soup.find_all("p") if p.get_text(strip=True)]

    # Extract author
    author_tag = soup.find("meta", {"name": "author"})
    author = author_tag["content"] if author_tag else "Unknown"

    # Extract published date
    published_date = None
    date_meta = soup.find("meta", {"property": "article:published_time"}) or \
                soup.find("meta", {"name": "date"}) or \
                soup.find("meta", {"itemprop": "datePublished"}) or \
                soup.find("time")

    if date_meta:
        if date_meta.has_attr("content"):
            published_date = date_meta["content"]
        elif date_meta.has_attr("datetime"):
            published_date = date_meta["datetime"]
        else:
            published_date = date_meta.get_text(strip=True)

    result = {
        "title": title,
        "url": url,
        "author": author,
        "published_date": published_date,
        "paragraphs": paragraphs
    }

    print(json.dumps(result, indent=2))

# Example usage
scrape_with_bs4("https://www.ndtvprofit.com/business/atm-usage-cost-revision-comes-into-effect-rs-23-per-withdrawal-after-free-limit")


{
  "title": "ATM Usage Cost Revision Comes Into Effect",
  "url": "https://www.ndtvprofit.com/business/atm-usage-cost-revision-comes-into-effect-rs-23-per-withdrawal-after-free-limit",
  "author": "PTI",
  "published_date": "2025-05-01T11:17:55.428+05:30",
  "paragraphs": [
    "The RBI's instructions on revised ATM usage cost have come into effect from Thursday under which banks can charge Rs 23 per cash withdrawal once a customer exhausts the free permissible limit in a month.",
    "Earlier, banks were allowed to charge up to Rs 21 per such transaction.",
    "Customers are eligible for five free transactions (inclusive of financial and non-financial transactions) every month from their own bank Automated Teller Machines (ATMs).",
    "They are also eligible for free transactions (inclusive of financial and non-financial transactions) from other bank ATMs -- three transactions in metro centres and five in non-metro centres.",
    "On March 28, the Reserve Bank of India (RBI) had is

news paper 3d

In [18]:
import json
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from newspaper import Article

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}

COOKIES = {
    'name_of_the_consent_cookie': 'value_indicating_consent',
}

def scrape_with_selenium_and_newspaper(url):
    # Set up headless browser with user-agent
    options = Options()
    options.add_argument("--headless")
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")

    driver = webdriver.Chrome(options=options)

    # Load the page
    driver.get(url)

    # Add cookies if needed (domain must match)
    for name, value in COOKIES.items():
        driver.add_cookie({
            'name': name,
            'value': value,
            'domain': '.' + url.split('/')[2]  # extract domain from URL
        })

    # Reload with cookies applied
    driver.get(url)
    time.sleep(3)  # Let the page load

    # Get fully rendered HTML
    html = driver.page_source
    driver.quit()

    # Use newspaper3k to parse the HTML
    article = Article(url)
    article.set_html(html)
    article.parse()

    paragraphs = [p.strip() for p in article.text.split("\n") if p.strip()]

    result = {
        "title": article.title,
        "url": url,
        "authors": article.authors,
        "published_date": article.publish_date.isoformat() if article.publish_date else "Unknown",
        "paragraphs": paragraphs
    }

    print(json.dumps(result, indent=2))

# Example usage
scrape_with_selenium_and_newspaper("https://www.ndtvprofit.com/business/atm-usage-cost-revision-comes-into-effect-rs-23-per-withdrawal-after-free-limit")


{
  "title": "ATM Usage Cost Revision Comes Into Effect",
  "url": "https://www.ndtvprofit.com/business/atm-usage-cost-revision-comes-into-effect-rs-23-per-withdrawal-after-free-limit",
  "authors": [],
  "published_date": "Unknown",
  "paragraphs": [
    "The RBI has, from time to time, issued various instructions on the number of free ATM transactions and maximum charges that can be levied on a customer beyond the mandatory free transactions. Instructions have also been issued by the RBI on interchange fee structure for ATM transactions."
  ]
}


In [19]:
import json
import requests
import trafilatura

def scrape_with_trafilatura(url):
    try:
        # Fetch the raw HTML from the URL
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        html = response.text
    except Exception as e:
        print(f"Failed to fetch the page: {e}")
        return

    # Extract structured content with metadata
    downloaded = trafilatura.extract(html, include_comments=False, include_tables=False, output_format='json')

    if downloaded is None:
        print("Failed to extract content with trafilatura.")
        return

    data = json.loads(downloaded)

    # Extract clean paragraphs
    paragraphs = data['text'].split('\n') if 'text' in data and data['text'] else []

    result = {
        "title": data.get("title", "Unknown"),
        "url": url,
        "authors": data.get("author", []),
        "published_date": data.get("date", "Unknown"),
        "paragraphs": [p.strip() for p in paragraphs if p.strip()]
    }

    print(json.dumps(result, indent=2, ensure_ascii=False))

# Example usage
scrape_with_trafilatura("https://www.ndtvprofit.com/business/atm-usage-cost-revision-comes-into-effect-rs-23-per-withdrawal-after-free-limit")

{
  "title": "Unknown",
  "url": "https://www.ndtvprofit.com/business/atm-usage-cost-revision-comes-into-effect-rs-23-per-withdrawal-after-free-limit",
  "authors": [],
  "published_date": "Unknown",
  "paragraphs": [
    "ATM Usage Cost Revision Comes Into Effect",
    "The usage cost has been raised by Rs 2 to Rs 23 per withdrawal after free monthly usage limit.",
    "The RBI's instructions on revised ATM usage cost have come into effect from Thursday under which banks can charge Rs 23 per cash withdrawal once a customer exhausts the free permissible limit in a month.",
    "Earlier, banks were allowed to charge up to Rs 21 per such transaction.",
    "Customers are eligible for five free transactions (inclusive of financial and non-financial transactions) every month from their own bank Automated Teller Machines (ATMs).",
    "They are also eligible for free transactions (inclusive of financial and non-financial transactions) from other bank ATMs -- three transactions in metro cent